# Count matrix
This script generates a table that has columns of samples and rows of guide counts.

VERY IMPORTANT: to only keep ABE or CBE guides in ABE/ CBE subpool files!

Kexin Dong 

Dec 2, 2025

### Load library and config file

In [2]:
import numpy as np 
import pandas as pd
import os
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
import re
import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42   
mpl.rcParams['ps.fonttype'] = 42 
mpl.rcParams['text.usetex'] = False

Load the **config file** (sample manifest from STEP3) and the **library file** that maps every `gRNA_id` to its target gene and classification.

> **Heads-up:** the library path here is a placeholder (`.../FINAL_focused_library.csv`). Point it at your own library CSV before running.


In [ ]:
config = pd.read_csv('CONFIG_BALL_VALIDATION_SCREEN.txt', sep=' ')
df = pd.read_csv('.../FINAL_focused_library.csv')

## Build human-readable sample IDs

The raw NGS sample names (`D25-xxxxx`) are not easy to read. We build a parallel list of friendly IDs that label each lane by **tissue → animal → lane**, e.g. `spleen1-1`, `spleen1-2`, `bm1-1`, ….

The outer loop is over animal index (5 animals), the middle over tissue (`spleen`, `bm`, `men`), the inner over the two NGS lanes (`-1` / `-2`).


In [ ]:
id_in_vivo = []
for y in range(5):
    for x in ['spleen','bm','men']:
        for z in range(2):
            id_in_vivo.append(f'{x}{y+1}-{z+1}')
id_in_vivo

## Map raw NGS IDs to friendly IDs

For the **ABE × BC** subpool, list the raw NGS sample IDs in a deliberate order (library → input → in-vitro d5 → in-vitro d15 (5 reps × 2 lanes) → in-vivo tissues), then build the matching list of friendly IDs (`lib-1`, `input-1`, `d5-1`, `d15-rep1-1`, …, `spleen1-1`, …).

The final `ABE_BC` dict maps `D25-xxxxx_guide_split_BARCODES` → friendly ID. We'll use this to rename count columns in the next step.

Repeat this cell pattern for **ABE × EPO**, **CBE × BC**, and **CBE × EPO** if you have those subpools.


In [ ]:
samp_ABE_BC = [
    'D25-13337-1-7331E','D25-13337-2-7331E', #library

    'D25-13245-1-7331E','D25-13245-2-7331E', #input

    'D25-13251-1-7331E','D25-13251-2-7331E', #in vitro d5

    'D25-13317-1-7331E','D25-13317-2-7331E', #in vitro d15
    'D25-13788-1-7331E','D25-13788-2-7331E',
    'D25-13789-1-7331E','D25-13789-2-7331E',
    'D25-13790-1-7331E','D25-13790-2-7331E',
    'D25-13321-1-7331E','D25-13321-2-7331E',

    'D25-13278-1-7331E','D25-13278-2-7331E', #spleen1
    'D25-13279-1-7331E','D25-13279-2-7331E', #bm1
    'D25-13280-1-7331E','D25-13280-2-7331E', #men1
    'D25-13281-1-7331E','D25-13281-2-7331E',
    'D25-13282-1-7331E','D25-13282-2-7331E',
    'D25-13283-1-7331E','D25-13283-2-7331E',
    'D25-13284-1-7331E','D25-13284-2-7331E',
    'D25-13285-1-7331E','D25-13285-2-7331E',
    'D25-13286-1-7331E','D25-13286-2-7331E',
    'D25-13287-1-7331E','D25-13287-2-7331E',
    'D25-13288-1-7331E','D25-13288-2-7331E',
    'D25-13289-1-7331E','D25-13289-2-7331E',
    'D25-13290-1-7331E','D25-13290-2-7331E',
    'D25-13291-1-7331E','D25-13291-2-7331E',
    'D25-13292-1-7331E','D25-13292-2-7331E',
]
samp_ABE_BC = [f'{x}_guide_split_BARCODES' for x in samp_ABE_BC]
id_abe_bc_d15 = []
for y in range(5):
    for z in range(2):
        id_abe_bc_d15.append(f'd15-rep{y+1}-{z+1}')
id_abe_bc_d15

samp_id_ABE_BC = [
    'lib-1','lib-2',
    'input-1','input-2',
    'd5-1','d5-2',
] + id_abe_bc_d15 + id_in_vivo

ABE_BC = dict(zip(samp_ABE_BC, samp_id_ABE_BC))
ABE_BC

## Annotate with gene + classification

`guide_gene` maps each `gRNA_id` to its target gene name, `class_gene` to its classification (e.g. `'non-targeting control'`, `'safe-targeting control'`, or a real target). These are used later to (a) attach a `gene` column to the count matrix, and (b) collapse all control sgRNAs under the single gene name `controlsgRNA` so MAGeCK treats them as one negative-control group.


In [ ]:
#and then add in the gene information as well
guide_gene = dict(zip(df['gRNA_id'], df['gene_name_m']))
class_gene = dict(zip(df['gRNA_id'], df['classification']))

## `count_matrix` — build the MAGeCK-ready table

This function does four things for one subpool:

1. **Load** every per-sample count CSV from STEP4's `counts/` folder and pull out the chosen count column (default `bc_count`; alternatives are `matched_guide_count` or `total_guide_count`).
2. **Concatenate** them column-wise, rename to friendly sample IDs, dedupe the repeated `Guide_ID` column.
3. **Attach the `gene` column** using `guide_gene` + `class_gene`, collapsing controls to `controlsgRNA`. Final column order: `sgRNA`, `gene`, then one column per sample.
4. **Merge NGS lanes** — sums every consecutive pair of columns (e.g. `spleen1-1 + spleen1-2 → spleen1`), so the output has one column per biological replicate.

Choose `bc_count` for the most lenient (highest) counts, or `matched_guide_count` for stricter (protospacer + barcode both correct) counts. See [STEP4 output reference](../../README.md) for definitions.

> **Heads-up:** set `fp` to your local STEP4 `counts/` folder before running.


In [ ]:
fp = '.../counts'
chosen_count_column = 'bc_count' #can also choose "matched_guide_count" or "total_guide_count"
df_holder = []

def count_matrix(dictionary, samp, samp_id):
    df_holder = []
    for i, k in enumerate(samp):
        aa = pd.read_csv(f'{fp}/{k}_count_df.csv')
        aa = aa[['Guide_ID', chosen_count_column]]
        s = dictionary[k]
        aa = aa.rename(columns = {chosen_count_column:f'{s}'})

        if i>0:
            aa = aa.rename(columns = {'Guide_ID':'Guide_ID_x'})
        df_holder.append(aa)

    join_out = pd.concat(df_holder, axis=1)
    sample_id_expanded_to_add = []
    for x in samp_id:
        if x in join_out.columns:
            sample_id_expanded_to_add.append(x)
    join_out = join_out[['Guide_ID'] + sample_id_expanded_to_add]
    join_out = join_out.loc[:, ~join_out.columns.duplicated()]

    genes = []
    for i, val in join_out.iterrows():
        k = val['Guide_ID']
        #print(k)
        gene = guide_gene[k]
        #make all the neutral guides have the same name
        class_ = class_gene[k]
        if class_ in ['non-targeting control','safe-targeting control']:
            gene = 'controlsgRNA'
        genes.append(gene)
    join_out['gene'] = genes
    join_out = join_out.rename(columns = {'Guide_ID':'sgRNA'})
    cols = ['sgRNA', 'gene'] + [c for c in join_out.columns if c not in ['sgRNA', 'gene']]
    join_out = join_out[cols]

    samps_to_merge = [
            [samp_id[i], samp_id[i+1]]
            for i in range(0, len(samp_id), 2)
        ]
    new_sample_ids = np.unique([x.rsplit("-", 1)[0] for x in samp_id])
    # print(new_sample_ids)
    for i,x in enumerate(samps_to_merge):
        join_out[new_sample_ids[i]] = join_out[x].sum(axis=1)
        join_out = join_out.drop(columns = x)
    return join_out

## Apply + filter to ABE-only guides

Build the count matrix for the ABE × BC subpool, then **drop all CBE guides** so MAGeCK only sees the editor-relevant guides for this comparison. The `Editor` column on the library CSV is the source of truth here.

> **VERY IMPORTANT:** repeat this filter step for every subpool — keep ABE guides in ABE files, CBE guides in CBE files. Mixing editors will silently destroy the LFC analysis.


In [ ]:
ABE_BC_COUNTS = count_matrix(ABE_BC, samp_ABE_BC, samp_id_ABE_BC)
ABE = df[df['Editor'] == 'ABE']
ABE_ids = ABE['gRNA_id'].tolist()
ABE_BC_COUNTS = ABE_BC_COUNTS[ABE_BC_COUNTS['sgRNA'].isin(ABE_ids)].reset_index(drop=True)

In [ ]:
#and finally save it for running on mageck
ABE_BC_COUNTS.to_csv('ABE_BC_COUNTS.txt', sep='\t', index=False)